# S3 through Trino: STS and impersonation

Every byte of data on this platform lives in Ceph S3, and Ceph alone decides who reads or writes a file. Every table is modelled in **Meta Catalog**: its provisioner makes the table's folder (`s3://dataplatform/lakehouse/<domain>/<subdomain>/<layer>/<table>/`) and registers the table in the Hive Metastore, and the table's own policy says which users and groups may read or write it (a member of the admins' group authorizes it). A grant holds twice: **Trino** refuses a statement on a table you were not granted (the Open Policy Agent decides), and **Ceph** refuses its files. Nobody holds a permanent S3 key: access goes through **STS roles**, one per user and group a policy names (`usr-<user>`, `grp-<group>`), and each role may only touch the folders of the tables granted to it. In the example domain `retail`, `the engineers' group` reads and writes every table and `the analysts' group` reads the gold ones.

When you query through Trino, this is what happens for each query:

1. You log in to Trino with Keycloak (OAuth 2). Trino knows your user name; your groups come from the file that `iam-sync` keeps up to date from Keycloak.
2. Trino asks the **Open Policy Agent** whether you may run the statement: a table you were not granted is refused here, and you do not see it in listings.
3. Trino's **S3 security mapping** (written by `iam-sync` from the same grants) has a rule for you per folder granted to you or to your groups: for each file, Trino takes the role of yours granted its folder (`rw` before `ro`, your own before your groups'). One query may therefore use tables granted to you and to your groups together. Anywhere else: your own role if a table grants you something, else your first group's. The extra credential `s3_role` would pin one role for the whole session instead.
4. Trino calls Ceph's STS **as itself** (the RGW user `trino`, which has no permission of its own) and asks to **assume that role on your behalf** (`AssumeRole`, session name = your user name). Ceph checks that the role trusts `trino`, and hands back temporary keys that last a few hours.
5. Trino reads and writes the table files with those keys. Anything outside the role's policy is refused by Ceph, whatever SQL you wrote.

That is the *impersonation*: Trino never uses a key of its own, it borrows your group's identity for the duration of your query. Below, the same thing done by hand with STS, so you can see what Trino does for you.

The notebook uses only your own login, once, at the top. Nothing here needs an admin.

## 1. The Trino client

The Python client is not part of the notebook image; `--user` puts it in your home volume, which survives restarts.

In [ ]:
%pip install --quiet --user trino boto3

## 2. Your token

You logged in to JupyterHub with Keycloak, and the hub handed that login's token to this server as `KEYCLOAK_TOKEN` (valid 12 hours). Everything below uses it, for Trino and for STS: no further login. If it has expired, stop and start your server from the hub's control panel (File > Hub Control Panel), or run the fallback below: a device login of the client `data-cli` (the cell prints a link; open it, log in and confirm).

In [ ]:
import base64, hashlib, json, os, secrets, sys, time, urllib.error, urllib.parse, urllib.request

ISSUER = os.environ["KEYCLOAK_ISSUER"]

def claims_of(t):
    return json.loads(base64.urlsafe_b64decode(t.split(".")[1] + "==").decode())

def post(url, **form):
    req = urllib.request.Request(url, urllib.parse.urlencode(form).encode())
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        return json.load(e)

def device_login():
    # PKCE: the client data-cli wants it (S256), in the device flow too
    verifier = secrets.token_urlsafe(64)
    challenge = base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).decode().rstrip("=")
    dev = post(f"{ISSUER}/protocol/openid-connect/auth/device", client_id="data-cli", scope="openid profile",
               code_challenge=challenge, code_challenge_method="S256")
    if "device_code" not in dev:
        raise SystemExit(f"login failed: {dev.get('error_description', dev)}")
    print("Open this link, log in and confirm:\n   ", dev["verification_uri_complete"], flush=True)
    deadline = time.time() + dev.get("expires_in", 600)
    while True:
        time.sleep(dev.get("interval", 5))
        tok = post(f"{ISSUER}/protocol/openid-connect/token", grant_type="urn:ietf:params:oauth:grant-type:device_code",
                   client_id="data-cli", device_code=dev["device_code"], code_verifier=verifier)
        if "access_token" in tok:
            print()
            return tok["access_token"]
        if tok.get("error") != "authorization_pending":
            raise SystemExit(f"login failed: {tok.get('error_description', tok)}")
        print(f"\r   waiting for you ... {int(deadline - time.time())} s left", end="", flush=True)

token = os.environ.get("KEYCLOAK_TOKEN", "")
if token and claims_of(token)["exp"] > time.time() + 60:
    print("using the token of your JupyterHub login")
else:
    token = device_login()
claims = claims_of(token)
print(f"you are {claims['preferred_username']} with groups {claims.get('groups')}; "
      f"the token is good for {int((claims['exp'] - time.time()) / 60)} more minutes")

## 3. Trino with that token

The token goes to Trino as a bearer JWT (Trino checks its signature against Keycloak, and its audience `trino`). No second login: Trino takes your name and, from `iam-sync`'s file, your groups. No role to choose either: for each file, Trino uses the role of yours granted its folder.

In [ ]:
from trino.dbapi import connect
from trino.auth import JWTAuthentication

TRINO = os.environ["TRINO_URL"].removeprefix("https://")
# no role to pass: Trino picks, for each file, the role of yours granted its folder
conn = connect(host=TRINO, port=443, http_scheme="https", auth=JWTAuthentication(token), catalog="lakehouse")
cur = conn.cursor()

def sql(statement):
    """Run one statement and return its rows (prints the reason when it is refused)."""
    try:
        cur.execute(statement)
        return cur.fetchall()
    except Exception as e:  # Trino's own refusal, or Ceph's wrapped in the error message
        print("refused:", str(e).split("\n")[0][:300])
        return None

sql("SELECT 42")

## 4. Who Trino thinks you are

`current_groups()` comes from Keycloak (through `iam-sync`). Which tables you may use, and which roles you have for their files, comes from the tables' policies in Meta Catalog: a user nothing grants, directly or through a group, reads no table and gets no S3 credentials from Trino.

In [ ]:
user, groups = sql("SELECT current_user, current_groups()")[0]
print("user:", user, " groups:", groups)
BUCKET, DOMAIN = os.environ["S3_BUCKET"], "retail"   # the example domain (subsystems/dataplatform/metacatalog)
ENGINEERS, ANALYSTS, ADMINS = os.environ["ENGINEER_GROUP"], os.environ["ANALYST_GROUP"], os.environ["ADMIN_GROUP"]
group = ENGINEERS if ENGINEERS in groups else sorted(groups)[0]
# what the tables of retail grant: the engineers write every table, the analysts read the gold ones
mine = f"lakehouse/{DOMAIN}/customers/silver/customers" if group == ENGINEERS else f"lakehouse/{DOMAIN}/analytics/gold/daily_revenue"
table = f"lakehouse.{DOMAIN}_customers_silver.customers" if group == ENGINEERS else f"lakehouse.{DOMAIN}_analytics_gold.daily_revenue"
# a table your group is not granted: the analysts' own of the smoke test, or a silver table of retail
theirs, their_table = (("lakehouse/smoke/test/gold/ana_t", "lakehouse.smoke_test_gold.ana_t") if group == ENGINEERS
                       else (f"lakehouse/{DOMAIN}/customers/silver/customers", f"lakehouse.{DOMAIN}_customers_silver.customers"))
other_group = next(g for g in [ENGINEERS, ANALYSTS, ADMINS] if g not in groups)
print(f"your role: grp-{group} -> s3://{BUCKET}/{mine}/   (not granted: s3://{BUCKET}/{theirs}/)")
print("the tables you see:", sql(f"SELECT table_schema || '.' || table_name FROM lakehouse.information_schema.tables WHERE table_schema <> 'information_schema'"))

## 5. Write and read a table

Meta Catalog made the table; you only write and read its data. The `INSERT` writes Parquet files to S3 with the temporary keys of `grp-<group>`: this is the impersonated write. (A member of `the analysts' group` only reads: the cell reads a gold table instead.) The table here is an Iceberg one, so rows can be deleted again; a `parquet` or `avro` table takes `SELECT` and `INSERT` only.

In [ ]:
if group == ENGINEERS:
    sql(f"""INSERT INTO {table} VALUES
        (-1, 'notebook-demo {user} a', 'a@notebook.invalid', 'XX'),
        (-2, 'notebook-demo {user} b', 'b@notebook.invalid', 'XX')""")
    print(sql(f"SELECT customer_id, full_name FROM {table} WHERE full_name LIKE 'notebook-demo {user}%' ORDER BY 1"))
else:
    print(sql(f"SELECT * FROM {table} LIMIT 5"))

Where the files are: Trino exposes the path of each data file. They are in the table's folder.

In [ ]:
for (path,) in sql(f'SELECT DISTINCT "$path" FROM {table}') or []:
    print(path)

## 6. What Trino refuses, and what Ceph refuses

Only Meta Catalog creates, changes or drops a schema or a table: Trino refuses it to everybody else, even next to a table you may write. A table your group is not granted is refused by Trino too, before any file is touched. Ceph refuses the same on its side, to whoever does not go through Trino: the next section lists that table's folder with your own keys.

In [ ]:
sql(f"CREATE TABLE {table.rsplit('.', 1)[0]}.my_own (x int)")   # refused by Trino: Meta Catalog's provisioner only
sql(f"SELECT count(*) FROM {their_table}")                       # refused by Trino: the table is not granted to you

## 7. The same by hand: STS with your own token

Trino does an `AssumeRole` with its own identity. Directly, you do an `AssumeRoleWithWebIdentity`: you hand Ceph your Keycloak token, and Ceph checks that the token's `groups` claim contains the role's group. This is what `make dataplatform-s3-login` and every Spark job do. The token is the one of the login above.

Exchange the token for the temporary keys of your role (`AWS_ENDPOINT_URL` is set in the notebook: boto3 talks to Ceph). Then list your table's folder with those keys.

In [ ]:
import boto3

ACCOUNT = os.environ["RGW_ACCOUNT_ID"]       # the platform's RGW IAM account: every role is in it
sts = boto3.client("sts")                   # directly at STS you name the role: its ARN
creds = sts.assume_role_with_web_identity(RoleArn=f"arn:aws:iam::{ACCOUNT}:role/grp-{group}", RoleSessionName=user,
                                          WebIdentityToken=token, DurationSeconds=3600)["Credentials"]
print("temporary key", creds["AccessKeyId"], "until", creds["Expiration"])
s3 = boto3.client("s3", aws_access_key_id=creds["AccessKeyId"], aws_secret_access_key=creds["SecretAccessKey"],
                  aws_session_token=creds["SessionToken"])
for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{mine}/", MaxKeys=10).get("Contents", []):
    print(f"{o['Size']:>8}  s3://{BUCKET}/{o['Key']}")
# the role reaches the folders of its tables only: another table's, or the bucket's root, is refused
from botocore.exceptions import ClientError
for prefix in (f"{theirs}/", ""):
    try:
        s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix, MaxKeys=1)
        print(f"listing s3://{BUCKET}/{prefix}: allowed")
    except ClientError as e:
        print(f"listing s3://{BUCKET}/{prefix}: {e.response['Error']['Code']}")

And the refusal, one level down: the other group's role does not trust your token (its `groups` claim lacks that group), so STS itself says no. You never get keys for it.

In [ ]:
try:
    sts.assume_role_with_web_identity(RoleArn=f"arn:aws:iam::{ACCOUNT}:role/grp-{other_group}", RoleSessionName=user, WebIdentityToken=token)
except ClientError as e:
    print("STS refused grp-" + other_group + ":", e.response["Error"]["Code"])

## 8. Clean up

Only the demo's own rows go: the table is Meta Catalog's.

In [ ]:
if group == ENGINEERS:
    sql(f"DELETE FROM {table} WHERE full_name LIKE 'notebook-demo {user}%'")